In [1]:
!pip install mcp
!pip install openai
!pip install asyncio
!pip install google-cloud-storage
!pip install gradio
!pip install edge-tts
!pip install nest_asyncio

In [2]:
from google.colab import userdata
from gradio import gradio as gr
import os
import edge_tts

In [3]:
from openai import OpenAI # Import the OpenAI class
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

### rag_mcp_server.py

This file defines a Micro-Agent Communication Protocol (MCP) server that exposes a retrieve_document tool. The main purpose of this tool is to simulate a Retrieval-Augmented Generation (RAG) system, where an agent can call this tool to "retrieve" relevant information based on a query.

Currently, it provides a hardcoded response for demonstration. In a real-world scenario, this tool would interface with a knowledge base or search engine to fetch actual documents.

In [4]:
!pip install langchain_core
!pip install langchain_openai

In [5]:
from google.cloud import storage
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(
    "/content/gcp-credentials.json"
)

client = storage.Client(
    project="agentic-chatbot-mcp",
    credentials=credentials
)

In [6]:
# Authenticate with Google Cloud to access GCS
from google.colab import auth
auth.authenticate_user()

# Implementing RAG

In [7]:
%%writefile /content/rag_core.py
import pandas as pd
from google.cloud import storage
from google.oauth2 import service_account
import os
import json
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
import sys
# Global variables for components
bucket = None
vector_store = None
embeddings = None
llm = None

def _initialize_rag_components():
    global bucket, vector_store, embeddings, llm
    if llm is not None:  # Check if already initialized
        return

    print("Initializing RAG components...")

    # --- GCS Auth and Data Download ---
    # Assuming gcp-credentials.json is already written to /content/ by the main process
    try:
        with open("/content/gcp-credentials.json", "r") as f:
            key_dict = json.load(f)
        credentials = service_account.Credentials.from_service_account_info(key_dict)
        client = storage.Client(
            project=key_dict["project_id"],
            credentials=credentials
        )
        bucket = client.bucket("my-agentic-chatbot-data-lake")
        bucket.blob("corpus_first_100.csv").download_to_filename("local_file.csv")
    except Exception as e:
        print(f"Error during GCS initialization: {e}")
        raise

    # --- Load into vector store ---
    df = pd.read_csv("local_file.csv")
    docs = [Document(page_content=str(row['text'])) for _, row in df.iterrows()]

    # Ensure OPENAI_API_KEY is available in the subprocess environment
    openai_api_key = os.environ.get("OPENAI_API_KEY")
    if not openai_api_key:
        raise ValueError("OPENAI_API_KEY not found in environment variables.")

    embeddings = OpenAIEmbeddings(model="text-embedding-3-large", openai_api_key=openai_api_key)
    vector_store = InMemoryVectorStore(embeddings)
    vector_store.add_documents(docs)

    # --- LLM ---
    llm = ChatOpenAI(model="gpt-4-turbo", temperature=0.8, max_tokens=250, openai_api_key=openai_api_key)
    print("RAG components initialized.")

# --- Prompts ---
chatbot_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a financial analyst expert with the goal of finding what drives financial markets and company performance.
You need to answer questions related to financial data in a polite manner to prospective clients.

Follow these rules:
1. Always answer based on the provided context only. Do NOT make up an answer.
2. If unsure, say "I don't have enough information to answer that"
3. Dealing with strict confidential information, divulge only information asked
4. Use bullet points if needed

Context:
{context}"""),
    ("human", "Question: {question}"),
])

condense_question_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given a chat history and a follow-up question, rephrase it as a standalone question."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

# --- Core RAG function ---
def ask_question(question: str, history=None) -> str:
    _initialize_rag_components() # Initialize components if not already done

    chat_history_for_llm = []
    if history:
        for turn in history:
            if isinstance(turn, dict):
                role = turn.get("role")
                content = turn.get("content", "")
                if role in ["user", "human"]:
                    chat_history_for_llm.append(HumanMessage(content=content))
                elif role in ["assistant", "ai"]:
                    chat_history_for_llm.append(AIMessage(content=content))

    standalone_question = question
    if chat_history_for_llm:
        try:
            standalone_question = (condense_question_prompt | llm).invoke({
                "question": question,
                "chat_history": chat_history_for_llm
            }).content
        except Exception as e:
            print(f"Error condensing question: {e}")

    retrieved_docs = vector_store.similarity_search(standalone_question, k=4)
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    messages = chatbot_prompt.format_messages(question=standalone_question, context=docs_content)
    payload = [{"role": "user" if m.type == "human" else "system" if m.type == "system" else "assistant",
                "content": m.content} for m in messages]
    return llm.invoke(payload).content
print("Th RAG Core file successfully saved and executed!", file=sys.stderr)

Overwriting /content/rag_core.py


#rag_mcp_server.py

In [8]:
%%writefile /content/rag_mcp_server.py
import sys
import os
import traceback

# Define a dedicated log file for internal server debugging
SERVER_INTERNAL_LOG = "/content/server_internal_debug.log"

def log_internal(message):
    with open(SERVER_INTERNAL_LOG, "a") as f:
        f.write(message + "\n")
    # Also try to print to stderr if it works, for immediate feedback during development
    print(message, file=sys.stderr, flush=True)

# Clear log on startup for a fresh debug session
with open(SERVER_INTERNAL_LOG, "w") as f:
    f.write("rag_mcp_server.py: Script execution started at " + str(os.getpid()) + "\n")

log_internal("rag_mcp_server.py: After initial log write, setting up...")

sys.path.insert(0, os.path.dirname(__file__))

try:
    from mcp.server.fastmcp import FastMCP
    log_internal("Successfully imported FastMCP.")
except Exception as e:
    log_internal(f"Error importing FastMCP: {e}")
    log_internal(traceback.format_exc())
    sys.exit(1) # Exit to make error clear

try:
    from rag_core import ask_question
    log_internal("Successfully imported rag_core.ask_question.")
except Exception as e:
    log_internal(f"Error importing rag_core: {e}")
    log_internal(traceback.format_exc())
    sys.exit(1) # Exit to make error clear

mcp = FastMCP("Agentic-RAG-Server")
log_internal("FastMCP instance created.")

@mcp.tool()
def search_knowledge_base(query: str) -> str:
    """
    Searches the internal knowledge base about financial markets and company data.
    Use this whenever the user asks about stock prices, company performance, economic indicators, or financial news.
    """
    log_internal(f"LOG: search_knowledge_base called for: {query}")
    try:
        result = ask_question(query)
        log_internal(f"LOG: search_knowledge_base returned result.")
        return result
    except Exception as e:
        log_internal(f"Error in search_knowledge_base: {e}")
        log_internal(traceback.format_exc())
        raise

log_internal("rag_mcp_server.py finished setup and configured FastMCP.")

# Call mcp.run() only when the script is executed directly
if __name__ == "__main__":
    log_internal("rag_mcp_server.py: Running mcp.run(transport=\"stdio\")...")
    mcp.run(transport="stdio")

print("Th server file successfully saved and executed!", file=sys.stderr)

Overwriting /content/rag_mcp_server.py


# Gradio app

In [9]:
%%writefile /content/gradio_app.py
import os
import asyncio
import openai
import edge_tts
import gradio as gr
from rag_core import ask_question

# --- TTS ---
async def _tts(text):
    communicate = edge_tts.Communicate(text, "en-US-AndrewNeural")
    path = "response.mp3"
    await communicate.save(path)
    return path

# --- Transcription ---
def transcribe_audio(audio_path):
    if not os.environ.get("OPENAI_API_KEY"):
        return "Error: OPENAI_API_KEY not set."
    client = openai.OpenAI()
    with open(audio_path, "rb") as f:
        transcript = client.audio.transcriptions.create(model="whisper-1", file=f)
    return transcript.text

# --- Gradio handlers ---
def predict_for_multimodal(message, history):
    if history is None:
        history = []
    rag_answer = ask_question(message, history)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": rag_answer})
    return history, ""

def voice_chat_handler(audio_path, history):
    status_msg = ""
    audio_output_path = None
    user_text = ""
    bot_text = ""
    if history is None:
        history = []
    if audio_path is None:
        return history, None, "No audio received."
    try:
        user_text = transcribe_audio(audio_path)
        if user_text.startswith("Error:"):
            return history, None, user_text
        bot_text = ask_question(user_text, history)
        audio_output_path = asyncio.run(_tts(bot_text))
        status_msg = "Response generated successfully!"
    except Exception as e:
        bot_text = f"Error: {e}"
        status_msg = f"Error: {e}"
    history.append({"role": "user", "content": user_text})
    history.append({"role": "assistant", "content": bot_text})
    return history, audio_output_path, status_msg

# --- Gradio UI ---
with gr.Blocks() as multimodal_app:
    gr.Markdown("## 🌍 Financial and Investment Analyst Chatbot")
    chatbot = gr.Chatbot(label="Chat History", type="messages", allow_tags=False)
    status_box = gr.Textbox(label="Voice Status", interactive=False)
    audio_out = gr.Audio(label="🔊 Voice Response", autoplay=True)

    with gr.Row():
        msg = gr.Textbox(placeholder="Type a question...", scale=4)
        audio_mic_input = gr.Audio(sources="microphone", type="filepath", label="Record with Microphone", scale=1)

    with gr.Row():
        audio_upload_input = gr.Audio(type="filepath", label="Upload Audio File", scale=2)
        submit_audio_upload_btn = gr.Button("Submit Uploaded Audio", scale=1)

    msg.submit(predict_for_multimodal, [msg, chatbot], [chatbot, msg])
    audio_mic_input.stop_recording(voice_chat_handler, [audio_mic_input, chatbot], [chatbot, audio_out, status_box])
    submit_audio_upload_btn.click(voice_chat_handler, [audio_upload_input, chatbot], [chatbot, audio_out, status_box])

multimodal_app.launch(share=True)
print("The Gradio file successfully saved and executed!", file=sys.stderr)

Overwriting /content/gradio_app.py


# MCP Client Session

In [10]:
import asyncio
import json
import os
import sys
import nest_asyncio
nest_asyncio.apply()

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from openai import OpenAI

openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

async def run_chatbot_agent(user_question: str):
    # Get current environment variables and add /content to PYTHONPATH
    env = os.environ.copy()
    python_path = env.get("PYTHONPATH", "")
    if "/content" not in python_path:
        if python_path:
            env["PYTHONPATH"] = "/content:" + python_path
        else:
            env["PYTHONPATH"] = "/content"

    server_params = StdioServerParameters(
        command=sys.executable,
        args=["/content/rag_mcp_server.py"], # Directly run the server script
        env=env # Pass the modified environment
    )
    subprocess_stderr_log = open("subprocess_stderr.log", "w", encoding="utf-8")
    # subprocess_stdout_log is opened but not passed to stdio_client as it's not supported.
    # The client's stdout/stderr are managed internally by stdio_client, and errlog captures only stderr.
    subprocess_stdout_log = open("subprocess_stdout.log", "w", encoding="utf-8")
    try:
        async with stdio_client(server_params, errlog=subprocess_stderr_log) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                tools_response = await session.list_tools()
                openai_tools = [
                    {
                        "type": "function",
                        "function": {
                            "name": t.name,
                            "description": t.description,
                            "parameters": t.inputSchema
                        }
                    } for t in tools_response.tools
                ]

                messages = [{"role": "user", "content": user_question}]
                response = openai_client.chat.completions.create(
                    model="gpt-4-turbo",
                    messages=messages,
                    tools=openai_tools
                )
                response_message = response.choices[0].message

                if response_message.tool_calls:
                    messages.append(response_message)
                    for tool_call in response_message.tool_calls:
                        print(f"🤖 Calling tool: {tool_call.function.name}")
                        args = json.loads(tool_call.function.arguments)
                        result = await session.call_tool(tool_call.function.name, arguments=args)
                        messages.append(
                            {
                                "role": "tool",
                                "tool_call_id": tool_call.id,
                                "name": tool_call.function.name,
                                "content": result.content[0].text
                            }
                        )
                    final = openai_client.chat.completions.create(
                        model="gpt-4o",
                        messages=messages
                    )
                    print("\n💬 Answer:\n", final.choices[0].message.content)
                else:
                    print("\n💬 Answer:\n", response_message.content)
    finally:
        subprocess_stderr_log.close()
        subprocess_stdout_log.close()

# Launch Gradio

In [11]:
!python /content/gradio_app.py

Th RAG Core file successfully saved and executed!
/content/gradio_app.py:59: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat History", type="messages", allow_tags=False)
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://475a536def0acb2865.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Initializing RAG components...
RAG components initialized.
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 3043, in block_thread
    time.sleep(0.1)
KeyboardInterrupt

During handling of the above

# Run MCP Agent

In [12]:
import asyncio
import os

# Set the PYTHONPATH for the current process before running asyncio.run
# This ensures that 'rag_mcp_server.mcp' can be found if the client is run directly.
python_path = os.environ.get("PYTHONPATH", "")
if "/content" not in python_path:
    if python_path:
        os.environ["PYTHONPATH"] = "/content:" + python_path
    else:
        os.environ["PYTHONPATH"] = "/content"

asyncio.run(run_chatbot_agent(user_question="What is a premium."))


💬 Answer:
 In financial and business contexts, the term "premium" can refer to several different concepts, depending on the specific area being discussed:

1. **Insurance Premium**: This is the amount that individuals or businesses must pay for an insurance policy. It can be paid in various frequencies such as monthly, quarterly, or annually. The premium covers the risk that the insurance company takes by insuring an individual or entity.

2. **Bond Premium**: This occurs when a bond is purchased for more than its par value. The premium over par can reflect factors such as a lower risk of the bond's issuer or interest rates that are lower than when the bond was issued.

3. **Option Premium**: In options trading, the premium is the price of an option, which the buyer pays to the seller. It is determined by factors including the underlying asset’s current price, strike price, time until expiration, and volatility.

4. **Premium in Marketing**: In promotional strategies, a premium might 

In [13]:
asyncio.run(run_chatbot_agent(user_question="What is a stock."))


💬 Answer:
 A stock represents a share in the ownership of a company and is a type of financial security. When someone buys a stock, they are purchasing a small part of the company – potentially gaining voting rights, dividends (a share of the company's earnings), and the ability to sell their shares to someone else. Stocks are bought and sold on stock exchanges, and their prices can fluctuate based on a variety of factors, including the company’s performance, economic conditions, and market demand. Holding stocks is one way for investors to potentially earn returns, though it comes with certain risks.


#Checking to see if the python file were written

In [14]:
import os
for f in ["rag_core.py", "rag_mcp_server.py", "gradio_app.py"]:
    print(f, "true" if os.path.exists(f"/content/{f}") else " MISSING")

rag_core.py true
rag_mcp_server.py true
gradio_app.py true


### ISSUE IS WITH MCP.RUN(), RAG CORE AND SERVER RUNS FFIL AFTER DEEBUGGING WHICH INCLUDE IMPORTING BOTH FASTMCP AND RAG_CORE.ASK_QUEESTION BOTH IMPORTD SUCCCESSFULLY AAND THE FASTMCCP INSTANCEEE WAS CREATD. LOGS WRE CREAATED AND CONFIRMD THAT THE IMPORTS IN RAG_MCP_SERVER.PY WERE SUCCESSFUL. CONNECTION WAS ESTAABLISHED USING IF__MAIN__

In [15]:
os.listdir("/content")

['.config',
 '__pycache__',
 'gradio_app.py',
 '.gradio',
 'gcp-credentials.json',
 'server_internal_debug.log',
 'subprocess_stderr.log',
 'subprocess_stdout.log',
 'rag_mcp_server.py',
 'response.mp3',
 'local_file.csv',
 'rag_core.py',
 'sample_data']